***Group 59 - Additional Activity***

# Match Outcome Dashboard with full guide

This notebook helps you run the existing **Streamlit Match Outcome Dashboard** from Google Colab.

## What this notebook does
1. Upload your project files.
2. Check required files and filenames.
3. Install dependencies.
4. Train the two Random Forest models.
5. Start Streamlit and create a temporary public link.

**Important:** The training script and dashboard expect particular CSV/JSON filenames. If the required files are missing, the notebook will show which ones are missing. Do not create empty placeholder CSV files; the result files must contain real model evaluation results.

## Step 1 - Upload your project files
Upload all the files from your dashboard project folder. If possible, select all the files together.

In [ ]:
from google.colab import files
uploaded = files.upload()

print("\nUploaded files:")
for name in sorted(uploaded):
    print(" -", name)

Saving app.py to app.py
Saving config.toml to config.toml
Saving final_feature_sets.json to final_feature_sets.json
Saving README.md to README.md
Saving requirements.txt to requirements.txt
Saving results_01_logistic_regression.csv to results_01_logistic_regression.csv
Saving results_02_random_forest.csv to results_02_random_forest.csv
Saving results_03_decision_tree.csv to results_03_decision_tree.csv
Saving results_04_xgboost.csv to results_04_xgboost.csv
Saving results_baseline.csv to results_baseline.csv
Saving test_final.csv to test_final.csv
Saving test_final_odds.csv to test_final_odds.csv
Saving train_final.csv to train_final.csv
Saving train_final_odds.csv to train_final_odds.csv
Saving train_model.py to train_model.py

Uploaded files:
 - README.md
 - app.py
 - config.toml
 - final_feature_sets.json
 - requirements.txt
 - results_01_logistic_regression.csv
 - results_02_random_forest.csv
 - results_03_decision_tree.csv
 - results_04_xgboost.csv
 - results_baseline.csv
 - test_

## Step 2 - Inspect the uploaded files
This checks filenames only; it does not rename or change any files.

In [ ]:
import os
from pathlib import Path

os.chdir("/content")
print("Current folder:", os.getcwd())
print("\nFiles found:")
for path in sorted(Path("/content").iterdir()):
    if path.is_file():
        print(f"{path.name}  ({path.stat().st_size:,} bytes)")

Current folder: /content

Files found:
README.md  (1,032 bytes)
app.py  (15,973 bytes)
config.toml  (217 bytes)
final_feature_sets.json  (708 bytes)
requirements.txt  (54 bytes)
results_01_logistic_regression.csv  (193 bytes)
results_02_random_forest.csv  (182 bytes)
results_03_decision_tree.csv  (181 bytes)
results_04_xgboost.csv  (168 bytes)
results_baseline.csv  (414 bytes)
test_final.csv  (979,796 bytes)
test_final_odds.csv  (882,230 bytes)
train_final.csv  (2,823,272 bytes)
train_final_odds.csv  (2,531,439 bytes)
train_model.py  (1,248 bytes)


## Step 3 - Check the filenames required by the scripts

The supplied `train_model.py` expects:
- `train_final.csv`
- `test_final.csv`
- `train_final_odds.csv`
- `test_final_odds.csv`
- `final_feature_sets.json`
- Five result CSVs: baseline, logistic regression, random forest, decision tree, and XGBoost.

The dashboard also reads `results_all.csv` and `season_outcomes.csv`, which are created by `train_model.py`.

If files have names like `train_final (1).csv`, rename them to the expected name in the next cell. If any required result CSV is missing, the training script will need that real result file or needs to be adjusted to handle a missing comparison model.

Also upload `config.toml` (the theme file). The next cell moves it into the `.streamlit` folder, because Colab uploads are flattened.

In [ ]:
from pathlib import Path

# Colab renames duplicate uploads like "train_final (1).csv" - restore the exact names the scripts expect.
import re
for p in sorted(Path(".").glob("*")):
    m = re.match(r"^(.*?)\s*\(\d+\)(\.[A-Za-z0-9]+)$", p.name)
    if m:
        new_name = m.group(1) + m.group(2)
        if not Path(new_name).exists():
            p.rename(new_name); print(f"Renamed: {p.name} -> {new_name}")
        else:
            print(f"Both {p.name} and {new_name} exist; left unchanged.")

# The dashboard theme file must live in a folder called .streamlit (uploads are flattened into /content)
Path(".streamlit").mkdir(exist_ok=True)
if Path("config.toml").exists():
    Path("config.toml").replace(".streamlit/config.toml"); print("Moved config.toml -> .streamlit/config.toml")
print(".streamlit/config.toml present:", Path(".streamlit/config.toml").exists())

required = [
    "app.py",
    "train_model.py",
    "requirements.txt",
    "final_feature_sets.json",
    "train_final.csv",
    "test_final.csv",
    "train_final_odds.csv",
    "test_final_odds.csv",
    "results_baseline.csv",
    "results_01_logistic_regression.csv",
    "results_02_random_forest.csv",
    "results_03_decision_tree.csv",
    "results_04_xgboost.csv",
]
missing = [name for name in required if not Path(name).exists()]

print("\nMissing required files:")
if missing:
    for name in missing:
        print(" -", name)
else:
    print("None — required files are present.")

Moved config.toml -> .streamlit/config.toml
.streamlit/config.toml present: True

Missing required files:
None — required files are present.


## Step 4 - Install the project dependencies

Run this only after uploading `requirements.txt`. Installation may take a few minutes.

In [ ]:
%cd /content
!pip install -q -r requirements.txt
!pip install -q streamlit

/content
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 47.3 MB/s eta 0:00:00


## Step 5 - Validate the input files

This checks that the input data and feature list can be opened, and checks the result CSV files required by the training script. If a result CSV is missing, stop here and upload it or update the project code; don't create an empty file.

In [ ]:
import json
import pandas as pd
from pathlib import Path

must_exist = [
    "train_final.csv", "test_final.csv", "train_final_odds.csv", "test_final_odds.csv",
    "final_feature_sets.json",
    "results_baseline.csv", "results_01_logistic_regression.csv",
    "results_02_random_forest.csv", "results_03_decision_tree.csv",
    "results_04_xgboost.csv",
]
missing = [f for f in must_exist if not Path(f).exists()]
if missing:
    raise FileNotFoundError(
        "Cannot train yet. Missing files:\n- " + "\n- ".join(missing) +
        "\nUpload the real files before continuing."
    )

with open("final_feature_sets.json", encoding="utf-8") as f:
    feature_sets = json.load(f)

train = pd.read_csv("train_final.csv")
test = pd.read_csv("test_final.csv")
train_odds = pd.read_csv("train_final_odds.csv")
test_odds = pd.read_csv("test_final_odds.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Train with odds shape:", train_odds.shape)
print("No-odds feature count:", len(feature_sets["features_no_odds"]))
print("With-odds feature count:", len(feature_sets["features_with_odds"]))

for label, df, features in [
    ("train", train, feature_sets["features_no_odds"]),
    ("test", test, feature_sets["features_no_odds"]),
    ("train_odds", train_odds, feature_sets["features_with_odds"]),
]:
    missing_cols = [c for c in features + ["target"] if c not in df.columns]
    if missing_cols:
        print(f"Missing columns in {label}: {missing_cols}")
    else:
        print(f"{label}: required feature/target columns are present.")

Train shape: (19328, 23)
Test shape: (6651, 23)
Train with odds shape: (16783, 23)
No-odds feature count: 16
With-odds feature count: 18
train: required feature/target columns are present.
test: required feature/target columns are present.
train_odds: required feature/target columns are present.


## Step 6 - Train the models and generate dashboard tables

This executes existing `train_model.py`. It should create:
- `rf_no_odds.joblib`
- `rf_with_odds.joblib`
- `season_outcomes.csv`
- `results_all.csv`
- `predictions_test.csv`

The notebook stops if training fails, so you can inspect the error instead of continuing with an incomplete dashboard.

In [ ]:
%cd /content
!python train_model.py

/content
saved models and tables


## Step 7 - Check generated files

In [ ]:
from pathlib import Path

generated = [
    "rf_no_odds.joblib",
    "rf_with_odds.joblib",
    "season_outcomes.csv",
    "results_all.csv",
    "predictions_test.csv",
]
for name in generated:
    p = Path(name)
    print(f"{name}: {'OK' if p.exists() else 'MISSING'}" +
          (f" ({p.stat().st_size:,} bytes)" if p.exists() else ""))

rf_no_odds.joblib: OK (1,762,241 bytes)
rf_with_odds.joblib: OK (1,765,073 bytes)
season_outcomes.csv: OK (198 bytes)
results_all.csv: OK (874 bytes)
predictions_test.csv: MISSING


## Step 8 - Start the Streamlit dashboard

This starts Streamlit in the background and creates a temporary Cloudflare Tunnel URL. Keep this Colab runtime active while you use the dashboard.

In [ ]:
import subprocess, time, os
from pathlib import Path

# Start Streamlit in the background and save logs for troubleshooting.
log_file = open("/content/streamlit.log", "w")
streamlit_process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0", "--server.headless", "true"],
    cwd="/content",
    stdout=log_file,
    stderr=subprocess.STDOUT,
)
time.sleep(5)

print("Streamlit process running:", streamlit_process.poll() is None)
print("\nRecent Streamlit log:")
if Path("/content/streamlit.log").exists():
    print(Path("/content/streamlit.log").read_text(errors="replace")[-3000:])

Streamlit process running: True

Recent Streamlit log:
2026-10-11 06:32:39.360 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.81.125.71:8501




## Step 9 - Create a public link

In [ ]:
import subprocess, os, time

# Download cloudflared (Linux x86_64 Colab runtime).
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

# Run the tunnel in the background; its log goes to a file so it never blocks
tunnel_log = open("/content/tunnel.log", "w")
tunnel_process = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8501"],
                                  stdout=tunnel_log, stderr=subprocess.STDOUT)

import re
found_url = None
for _ in range(60):
    time.sleep(1)
    text = open("/content/tunnel.log", errors="replace").read()
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", text)
    if m:
        found_url = m.group(0)
        print("Open this dashboard URL:", found_url)
        break

if not found_url:
    print("No URL yet - rerun this cell or check /content/tunnel.log")

Open this dashboard URL: https://grill-internationally-continuous-arrangements.trycloudflare.com


## Troubleshooting

- **FileNotFoundError:** one or more required files were not uploaded or have a different name.
- **KeyError for a feature or `target`:** the CSV columns do not match the feature JSON.
- **Missing results CSV:** upload the real result file or modify `train_model.py` and `app.py` to handle unavailable model comparisons.
- **Streamlit error:** inspect `/content/streamlit.log` using the cell below.
- **Temporary URL stops working:** Colab runtime/tunnel may have stopped; rerun the start/tunnel steps.

Don't share confidential data through the public tunnel. The URL is temporary and accessible to anyone who has it.

In [ ]:
from pathlib import Path
log = Path("/content/streamlit.log")
print(log.read_text(errors="replace")[-6000:] if log.exists() else "No Streamlit log found yet.")

2026-10-11 06:32:39.360 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.81.125.71:8501




In [ ]:
%cd /content
import pandas as pd, json, joblib

feats  = json.load(open("final_feature_sets.json"))
test   = pd.read_csv("test_final.csv")
test_o = pd.read_csv("test_final_odds.csv")

parts = []
for name, model_file, df, cols in [
    ("No odds",   "rf_no_odds.joblib",   test,   feats["features_no_odds"]),
    ("With odds", "rf_with_odds.joblib", test_o, feats["features_with_odds"]),
]:
    m = joblib.load(model_file)
    p = m.predict_proba(df[cols])          # columns: 0 = Away, 1 = Draw, 2 = Home
    parts.append(pd.DataFrame({"features": name,
                               "p_away": p[:, 0], "p_draw": p[:, 1], "p_home": p[:, 2],
                               "pred": m.predict(df[cols]), "actual": df["target"].values}))

pd.concat(parts, ignore_index=True).to_csv("predictions_test.csv", index=False)
print("saved predictions_test.csv")

/content
saved predictions_test.csv
